# Predicción semanal de casos de Dengue en Perú (2000–2024)
## Regresión / Serie temporal

**Objetivo:** predecir el **número de casos de dengue por semana epidemiológica** para construir
un **sistema de alerta temprana** de brotes.

**Pregunta de investigación:** *¿cuánta historia conviene usar y con cuántas semanas de
anticipación podemos predecir de forma confiable?*

### Nota metodológica: dos conceptos de "ventana"
1. **Ventana de entrada `W`** (semanas pasadas usadas como *features*): el error **baja y se
   aplana** → codo de **rendimientos decrecientes**.
2. **Horizonte `H`** (semanas hacia el futuro que se predicen): el error **crece** → se elige el
   **mayor `H` con error aceptable** (alerta temprana).

> La idea de "el error crece con la ventana y se elige el codo" es rigurosa para el **horizonte
> `H`**, no para la ventana de entrada `W`. Por eso reportamos **ambos** análisis.

## 0. Librerías

In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'   # silenciar logs de TensorFlow
import sys; sys.path.append('../src')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pandas.plotting import autocorrelation_plot

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge, Lasso
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
def rmse(y, p): return np.sqrt(mean_squared_error(y, p))

## 1. Carga y agregación a serie temporal

El dataset crudo son **registros individuales** (1 fila = 1 caso). Para regresión lo agregamos
contando casos por semana epidemiológica.

In [2]:
from data import cargar_crudo

df = cargar_crudo()
df['ano'] = pd.to_numeric(df['ano'], errors='coerce')
df['semana'] = pd.to_numeric(df['semana'], errors='coerce')
df = df.dropna(subset=['ano', 'semana'])
df = df[df['semana'].between(1, 53)]

serie = (df.groupby(['ano', 'semana']).size()
           .rename('casos').reset_index()
           .sort_values(['ano', 'semana']).reset_index(drop=True))
serie['t'] = range(len(serie))
print('Puntos en la serie:', len(serie))
serie.head()

Puntos en la serie: 1305


,ano,semana,casos,t
0,2000,1,14,0
1,2000,2,28,1
2,2000,3,26,2
3,2000,4,27,3
4,2000,5,59,4


### 1.1 Validación del dato (el target es real, no inventado)

In [3]:
print('Suma de la serie     :', int(serie['casos'].sum()))
print('Nº de registros reales:', len(df))
print('¿Coinciden?           :', int(serie['casos'].sum()) == len(df))
faltan = []
for a in range(int(serie['ano'].min()), int(serie['ano'].max()) + 1):
    s = set(serie[serie['ano'] == a]['semana'])
    if s:
        faltan += [(a, w) for w in set(range(1, max(s) + 1)) - s]
print('Semanas faltantes     :', faltan if faltan else 'NINGUNA (serie continua)')

Suma de la serie     : 1029421
Nº de registros reales: 1029421
¿Coinciden?           : True
Semanas faltantes     : NINGUNA (serie continua)


## 2. Análisis Exploratorio de la serie temporal

### 2.1 Serie completa

In [4]:
plt.figure(figsize=(13, 4))
plt.plot(serie['t'], serie['casos'], color='#c0392b', lw=0.8)
plt.title('Casos de dengue por semana epidemiológica (2000–2024)')
plt.xlabel('Semana (índice continuo)'); plt.ylabel('Nº de casos')
plt.tight_layout(); plt.show()
print(serie['casos'].describe().round(1))

count     1305.0
mean       788.8
std       2172.9
min         11.0
25%         95.0
50%        223.0
75%        596.0
max      21628.0
Name: casos, dtype: float64


C:\Users\danie\AppData\Local\Temp\ipykernel_2920\3896149724.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


### 2.2 Tendencia anual (el brote 2023–2024)

In [5]:
casos_anio = df.groupby('ano').size()
ax = casos_anio.plot(marker='o', figsize=(10, 4), color='#1f77b4')
ax.set_title('Tendencia anual de casos de dengue')
ax.set_xlabel('Año'); ax.set_ylabel('Casos'); plt.tight_layout(); plt.show()
print(casos_anio.tail(6))

ano
2019     15287
2020     47932
2021     44791
2022     63217
2023    256641
2024    271531
dtype: int64


C:\Users\danie\AppData\Local\Temp\ipykernel_2920\2540339417.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set_xlabel('Año'); ax.set_ylabel('Casos'); plt.tight_layout(); plt.show()


### 2.3 Estacionalidad (perfil por semana epidemiológica)

In [6]:
perfil_sem = serie.groupby('semana')['casos'].mean()
ax = perfil_sem.plot(figsize=(10, 4), color='#9467bd', marker='.')
ax.set_title('Estacionalidad: promedio de casos por semana del año')
ax.set_xlabel('Semana epidemiológica (1–53)'); ax.set_ylabel('Casos promedio')
plt.tight_layout(); plt.show()
print('Semana pico (en promedio):', int(perfil_sem.idxmax()))

Semana pico (en promedio): 16


C:\Users\danie\AppData\Local\Temp\ipykernel_2920\2236416285.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


### 2.4 Autocorrelación (¿la serie es predecible?)

In [7]:
plt.figure(figsize=(10, 4))
autocorrelation_plot(serie['casos'])
plt.title('Función de autocorrelación (ACF)')
plt.xlim(0, 60); plt.tight_layout(); plt.show()
print('Autocorrelación con la semana previa (lag 1):',
      round(serie['casos'].autocorr(lag=1), 3))

Autocorrelación con la semana previa (lag 1): 0.982


C:\Users\danie\AppData\Local\Temp\ipykernel_2920\2255968827.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.xlim(0, 60); plt.tight_layout(); plt.show()


## 3. Metodología: las dos ventanas

Split temporal (entrenar con el pasado, probar con el futuro), correcto para series temporales.

In [8]:
v = serie['casos'].values.astype(float)

def construir(v, n_lags, h):
    """X = [t-n_lags, ..., t-1]  ->  y = valor en t+h-1 (h pasos adelante)."""
    X, y = [], []
    for i in range(n_lags, len(v) - h + 1):
        X.append(v[i - n_lags:i]); y.append(v[i + h - 1])
    return np.array(X), np.array(y)

def split_temporal(X, y, test_frac=0.2):
    n_test = int(len(X) * test_frac)
    return X[:-n_test], X[-n_test:], y[:-n_test], y[-n_test:]

def evaluar(modelo, n_lags, h):
    X, y = construir(v, n_lags, h)
    Xtr, Xte, ytr, yte = split_temporal(X, y)
    modelo.fit(Xtr, ytr); pred = modelo.predict(Xte)
    return {'R2': r2_score(yte, pred), 'RMSE': rmse(yte, pred),
            'MAE': mean_absolute_error(yte, pred)}, (yte, pred)

### 3.1 Ventana de ENTRADA `W` (¿cuánta historia usar?)

Fijamos el horizonte en 1 y variamos `W`. El error **baja y se aplana**; elegimos el **codo**.

In [9]:
Ws = [1, 2, 3, 4, 5, 6, 8, 10, 12]
curva_W = pd.DataFrame([{'W': w, **evaluar(Ridge(), w, 1)[0]} for w in Ws]).set_index('W')
print(curva_W.round(2))
plt.figure(figsize=(9, 4))
plt.plot(curva_W.index, curva_W['RMSE'], 'o-', color='#2980b9')
plt.title('Ventana de ENTRADA vs. error (el error BAJA y se aplana)')
plt.xlabel('W = nº de semanas pasadas usadas como entrada'); plt.ylabel('RMSE')
plt.tight_layout(); plt.show()

      R2    RMSE     MAE
W                       
1   0.96  864.29  378.45
2   0.97  733.19  309.64
3   0.98  670.51  278.36
4   0.98  668.70  276.35
5   0.98  668.22  275.56
6   0.98  668.59  277.97
8   0.98  659.99  271.15
10  0.98  659.05  271.54
12  0.98  660.19  273.36


C:\Users\danie\AppData\Local\Temp\ipykernel_2920\937542454.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [10]:
mejor_rmse = curva_W['RMSE'].min()
W_OPT = int(curva_W[curva_W['RMSE'] <= 1.02 * mejor_rmse].index.min())
print(f'Mejor RMSE = {mejor_rmse:.0f} | umbral 2% = {1.02*mejor_rmse:.0f}')
print(f'Ventana de entrada óptima (codo): W = {W_OPT} semanas')

Mejor RMSE = 659 | umbral 2% = 672
Ventana de entrada óptima (codo): W = 3 semanas


### 3.2 HORIZONTE `H` (¿cuántas semanas adelante?)

Fijamos `W = W_OPT` y variamos `H`. El error **crece**; elegimos el **mayor `H` aún pequeño**.

In [11]:
Hs = [1, 2, 3, 4, 6, 8, 10, 12]
curva_H = pd.DataFrame([{'H': h, **evaluar(Ridge(), W_OPT, h)[0]} for h in Hs]).set_index('H')
print(curva_H.round(3))
plt.figure(figsize=(9, 4))
plt.plot(curva_H.index, curva_H['RMSE'], 'o-', color='#c0392b')
plt.title('HORIZONTE de pronóstico vs. error (el error CRECE con la ventana)')
plt.xlabel('H = nº de semanas hacia el futuro'); plt.ylabel('RMSE')
plt.tight_layout(); plt.show()

       R2      RMSE       MAE
H                            
1   0.975   670.509   278.362
2   0.927  1150.438   504.107
3   0.832  1742.273   781.170
4   0.712  2283.417  1038.617
6   0.404  3283.579  1492.625
8   0.126  3976.654  1811.345
10 -0.079  4424.445  2038.904
12 -0.199  4664.231  2190.239


C:\Users\danie\AppData\Local\Temp\ipykernel_2920\1788709334.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [12]:
base = curva_H.loc[1, 'RMSE']
H_OPT = int(curva_H[curva_H['RMSE'] <= 1.3 * base].index.max())
print(f'RMSE(H=1) = {base:.0f} | umbral 1.3x = {1.3*base:.0f}')
print(f'Horizonte óptimo (codo): H = {H_OPT} semana(s)')
print(f'\n=> Configuración elegida: entrada W={W_OPT} semanas, horizonte H={H_OPT} semana(s)')

RMSE(H=1) = 671 | umbral 1.3x = 872
Horizonte óptimo (codo): H = 1 semana(s)

=> Configuración elegida: entrada W=3 semanas, horizonte H=1 semana(s)


## 4. Comparación de técnicas (con `W` y `H` óptimos)

Cinco técnicas clásicas — **Ridge, Lasso, Perceptrón (MLP), Random Forest, Gradient Boosting** —
más la técnica de referencia para series temporales, **LSTM**.

In [13]:
modelos = {
    'Ridge (L2)': Ridge(alpha=1.0),
    'Lasso (L1)': Lasso(alpha=1.0, max_iter=5000),
    'Perceptrón (MLP)': make_pipeline(StandardScaler(),
                          MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=800,
                                       random_state=RANDOM_STATE)),
    'Random Forest': RandomForestRegressor(n_estimators=300, n_jobs=2,
                                           random_state=RANDOM_STATE),
    'Gradient Boosting': GradientBoostingRegressor(random_state=RANDOM_STATE),
}
resultados, preds = {}, {}
for nombre, m in modelos.items():
    met, (yte, p) = evaluar(m, W_OPT, H_OPT)
    resultados[nombre] = met; preds[nombre] = (yte, p)
print('5 técnicas clásicas entrenadas.')

5 técnicas clásicas entrenadas.


### 4.1 LSTM (red recurrente, técnica de referencia)

In [14]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input
tf.random.set_seed(RANDOM_STATE); np.random.seed(RANDOM_STATE)

# El LSTM necesita datos escalados y con forma 3D (muestras, pasos, features).
X, y = construir(v, W_OPT, H_OPT)
Xtr, Xte, ytr, yte = split_temporal(X, y)
sx = StandardScaler().fit(Xtr); sy = StandardScaler().fit(ytr.reshape(-1, 1))
Xtr_s = sx.transform(Xtr).reshape(-1, W_OPT, 1)
Xte_s = sx.transform(Xte).reshape(-1, W_OPT, 1)
ytr_s = sy.transform(ytr.reshape(-1, 1)).ravel()

lstm = Sequential([Input((W_OPT, 1)), LSTM(32), Dense(1)])
lstm.compile(optimizer='adam', loss='mse')
lstm.fit(Xtr_s, ytr_s, epochs=80, batch_size=16, verbose=0)
p_lstm = sy.inverse_transform(lstm.predict(Xte_s, verbose=0)).ravel()

resultados['LSTM'] = {'R2': r2_score(yte, p_lstm), 'RMSE': rmse(yte, p_lstm),
                      'MAE': mean_absolute_error(yte, p_lstm)}
preds['LSTM'] = (yte, p_lstm)
print('LSTM entrenado.')

LSTM entrenado.


### 4.2 Tabla comparativa (6 técnicas)

In [15]:
tabla = pd.DataFrame(resultados).T[['R2', 'RMSE', 'MAE']].round(3).sort_values('RMSE')
tabla

,R2,RMSE,MAE
Ridge (L2),0.975,670.509,278.362
Lasso (L1),0.975,670.519,278.372
Perceptrón (MLP),0.972,708.368,297.367
LSTM,0.560,2816.683,914.999
Random Forest,0.444,3166.712,1053.587
Gradient Boosting,0.374,3359.238,1128.383


### 4.3 Comparación con el *baseline* ingenuo

In [16]:
base_pred = Xte[:, -1]    # último valor conocido (persistencia)
print('Baseline ingenuo (persistencia):')
print(f"  R2={r2_score(yte, base_pred):.3f}  RMSE={rmse(yte, base_pred):.0f}  "
      f"MAE={mean_absolute_error(yte, base_pred):.0f}")
print('\nUn buen modelo debe SUPERAR este baseline.')

Baseline ingenuo (persistencia):
  R2=0.959  RMSE=862  MAE=383

Un buen modelo debe SUPERAR este baseline.


### 4.4 Predicho vs. real (mejor modelo)

In [17]:
mejor = tabla.index[0]
yte_m, p_m = preds[mejor]
plt.figure(figsize=(12, 4))
plt.plot(yte_m, label='Real', color='#2c3e50')
plt.plot(p_m, label='Predicho', color='#e67e22', alpha=0.85)
plt.title(f'Predicción de casos semanales — {mejor} (W={W_OPT}, H={H_OPT})')
plt.xlabel('Semana (conjunto de prueba)'); plt.ylabel('Casos')
plt.legend(); plt.tight_layout(); plt.show()

C:\Users\danie\AppData\Local\Temp\ipykernel_2920\132339021.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(); plt.tight_layout(); plt.show()


## 5. Insights

> _Completar con los valores obtenidos:_

1. **Ventana de entrada:** bastan ~`W` semanas de historia; más no mejora (rendimientos
   decrecientes) → fuerte **autocorrelación a corto plazo** (lag-1 ≈ 0.98).
2. **Horizonte:** el error **crece** al predecir más lejos; el horizonte confiable es **`H`
   semana(s)** → capacidad de **alerta temprana**.
3. **LSTM no superó a los modelos lineales.** Aunque es la técnica de referencia para series
   temporales, en esta serie —con autocorrelación muy alta y horizonte de 1 semana— los modelos
   lineales (Ridge/Lasso), que se anclan al último valor, son superiores. El LSTM además sufre
   con el **brote 2023–2024** (magnitudes nunca vistas en el entrenamiento). *Insight relevante:
   más complejidad no siempre es mejor.*
4. **Los árboles (RF, GB) fallan** porque no extrapolan la tendencia del brote.
5. **El mejor modelo supera al baseline ingenuo** → aporta valor real.

## 6. Conclusiones

- El dataset es **adecuado para regresión / serie temporal**: predice casos con `H` semana(s) de
  anticipación usando `W` semanas de historia, superando al baseline.
- **Mejor modelo:** lineal regularizado (Ridge/Lasso), simple e interpretable.
- **Aplicación:** sistema de alerta temprana de brotes.
- **Trabajo futuro:** dar más contexto al LSTM (más variables/épocas), incorporar variables
  exógenas (clima: temperatura y lluvias) y predicción por departamento.